In [79]:
import pandas as pd
import numpy as np

import warnings 
warnings.filterwarnings('ignore')

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples,silhouette_score
from sklearn.preprocessing import MinMaxScaler , OneHotEncoder

In [80]:
df_amazon = pd.read_csv('datasets2/amazon_titles.csv')
df_netflix = pd.read_csv('datasets2/netflix_titles.csv')
df_HBO = pd.read_csv('datasets2/HBO_titles.csv')

In [81]:
df = pd.concat([df_netflix,df_amazon,df_HBO],axis=0)
df.head(5)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts20945,The Three Stooges,SHOW,The Three Stooges were an American vaudeville ...,1934,TV-PG,19,"['comedy', 'family', 'animation', 'action', 'f...",['US'],26.0,tt0850645,8.6,1092.0,15.424,7.6
1,tm19248,The General,MOVIE,"During America’s Civil War, Union spies steal ...",1926,NaN,78,"['action', 'drama', 'war', 'western', 'comedy'...",['US'],NaN,tt0017925,8.2,89766.0,8.647,8.0
2,tm82253,The Best Years of Our Lives,MOVIE,It's the hope that sustains the spirit of ever...,1946,NaN,171,"['romance', 'war', 'drama']",['US'],NaN,tt0036868,8.1,63026.0,8.435,7.8
3,tm83884,His Girl Friday,MOVIE,"Hildy, the journalist former wife of newspaper...",1940,NaN,92,"['comedy', 'drama', 'romance']",['US'],NaN,tt0032599,7.8,57835.0,11.270,7.4
4,tm56584,In a Lonely Place,MOVIE,An aspiring actress begins to suspect that her...,1950,NaN,94,"['thriller', 'drama', 'romance']",['US'],NaN,tt0042593,7.9,30924.0,8.273,7.6


In [82]:
df.columns

Index(['id', 'title', 'type', 'description', 'release_year',
       'age_certification', 'runtime', 'genres', 'production_countries',
       'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity',
       'tmdb_score'],
      dtype='object')

In [83]:
df_movies = df.drop_duplicates()
df_movies.duplicated().sum()

0

In [84]:
df_movies.drop(['description','age_certification'],axis=1,inplace=True)

In [85]:
df_movies

,id,title,type,release_year,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts20945,The Three Stooges,SHOW,1934,19,"['comedy', 'family', 'animation', 'action', 'f...",['US'],26.0,tt0850645,8.6,1092.0,15.424,7.6
1,tm19248,The General,MOVIE,1926,78,"['action', 'drama', 'war', 'western', 'comedy'...",['US'],NaN,tt0017925,8.2,89766.0,8.647,8.0
2,tm82253,The Best Years of Our Lives,MOVIE,1946,171,"['romance', 'war', 'drama']",['US'],NaN,tt0036868,8.1,63026.0,8.435,7.8
3,tm83884,His Girl Friday,MOVIE,1940,92,"['comedy', 'drama', 'romance']",['US'],NaN,tt0032599,7.8,57835.0,11.270,7.4
4,tm56584,In a Lonely Place,MOVIE,1950,94,"['thriller', 'drama', 'romance']",['US'],NaN,tt0042593,7.9,30924.0,8.273,7.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3289,tm1082718,Romeo Santos: Utopia Live from MetLife Stadium,MOVIE,2021,103,"['romance', 'music']",['PR'],NaN,NaN,NaN,NaN,8.425,8.1
3290,tm1067128,Algo Azul,MOVIE,2021,90,['comedy'],['PA'],NaN,tt9257620,5.9,50.0,1.400,2.0
3291,tm1121489,Entre Nos: What She Said,MOVIE,2021,28,['comedy'],[],NaN,tt15532762,NaN,NaN,NaN,NaN
3292,tm1121486,Entre Nos: The Winners 2,MOVIE,2021,28,['comedy'],[],NaN,tt15532736,NaN,NaN,NaN,NaN


In [86]:
df_movies['production_countries']

0       ['US']
1       ['US']
2       ['US']
3       ['US']
4       ['US']
         ...  
3289    ['PR']
3290    ['PA']
3291        []
3292        []
3293    ['US']
Name: production_countries, Length: 13132, dtype: object

# Worked with production_countries

In [87]:
df_movies['production_countries'] = df_movies['production_countries'].str.replace(r"\[",'',regex=True).str.replace(r"'",'',regex=True).str.replace(r"\]",'',regex=True)

In [88]:
df_movies['lead_prod_countries'] = df_movies['production_countries'].str.split(',').str[0]
df_movies['prod_countries_cnt'] = df_movies['production_countries'].str.split(',').str.len()
df_movies['lead_prod_countries'] = df_movies['lead_prod_countries'].replace("",np.nan)
df_movies['lead_prod_countries']

0        US
1        US
2        US
3        US
4        US
       ... 
3289     PR
3290     PA
3291    NaN
3292    NaN
3293     US
Name: lead_prod_countries, Length: 13132, dtype: object

# Worked with Generes

In [89]:
df_movies['genres'] = df_movies['genres'].str.replace('\[','',regex=True).str.replace("'",'',regex=True).str.replace('\]','',regex=True)
df_movies['main_genre'] = df_movies['genres'].str.split(',').str[0]
df_movies['main_genre'] = df_movies['main_genre'].replace('',np.nan)
df_movies['main_genre']

0              comedy
1              action
2             romance
3              comedy
4            thriller
            ...      
3289          romance
3290           comedy
3291           comedy
3292           comedy
3293    documentation
Name: main_genre, Length: 13132, dtype: object

In [90]:
df_movies.drop(['genres','production_countries'],axis=1,inplace=True)

In [91]:
pd.DataFrame(df_movies.columns)

,0
0,id
1,title
2,type
3,release_year
4,runtime
5,seasons
6,imdb_id
7,imdb_score
8,imdb_votes
9,tmdb_popularity


# Dropped Missing Values

In [92]:

df_movies.shape

(13132, 14)

In [93]:
df_movies.isnull().sum()

id                         0
title                      0
type                       0
release_year               0
runtime                    0
seasons                11030
imdb_id                  993
imdb_score              1393
imdb_votes              1414
tmdb_popularity          579
tmdb_score              2345
lead_prod_countries      931
prod_countries_cnt         0
main_genre               262
dtype: int64

In [94]:
df_movies.dropna(inplace=True)
df_movies.set_index('title',inplace=True)
df_movies.drop(['id','imdb_id'],axis=1,inplace=True)

In [95]:
df_movies.shape

(1473, 11)

In [96]:
df_movies

,type,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_countries,prod_countries_cnt,main_genre
title,,,,,,,,,,,
The Three Stooges,SHOW,1934,19,26.0,8.6,1092.0,15.424,7.6,US,1,comedy
What's My Line?,SHOW,1950,30,18.0,8.6,1563.0,87.392,6.9,US,1,reality
I Love Lucy,SHOW,1951,30,9.0,8.5,25944.0,17.088,8.1,US,1,comedy
Mister Rogers' Neighborhood,SHOW,1968,29,31.0,8.7,8675.0,8.747,4.7,US,1,fantasy
Lupin the Third,SHOW,1971,23,6.0,7.9,2116.0,45.829,8.0,JP,1,scifi
...,...,...,...,...,...,...,...,...,...,...,...
Level Playing Field,SHOW,2021,26,1.0,5.5,60.0,4.595,5.0,US,1,documentation
Os Ausentes,SHOW,2021,46,1.0,5.9,59.0,4.624,10.0,BR,1,action
Through Our Eyes,SHOW,2021,33,1.0,6.1,38.0,0.840,1.0,US,1,documentation


# Encoding Categorical Features:

In [98]:
dummies =  pd.get_dummies(df_movies[['type','lead_prod_countries','main_genre']],drop_first=True)
df_movies_dum = pd.concat([df_movies,dummies],axis=1)
df_movies_dum.drop(['type','lead_prod_countries','main_genre'],axis=1,inplace=True)

# Min Max Scaled

In [101]:
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_movies_dum)
df_scaled = pd.DataFrame(df_scaled,columns=df_movies_dum.columns)

df_scaled

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,prod_countries_cnt,lead_prod_countries_AT,lead_prod_countries_AU,...,main_genre_fantasy,main_genre_horror,main_genre_music,main_genre_reality,main_genre_romance,main_genre_scifi,main_genre_sport,main_genre_thriller,main_genre_war,main_genre_western
0,0.000000,0.118421,0.490196,0.8875,0.000548,0.016204,0.739130,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.181818,0.190789,0.333333,0.8875,0.000785,0.091811,0.663043,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.193182,0.190789,0.156863,0.8750,0.013075,0.017952,0.793478,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.386364,0.184211,0.588235,0.9000,0.004370,0.009189,0.423913,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.420455,0.144737,0.098039,0.8000,0.001064,0.048146,0.782609,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1468,0.988636,0.164474,0.000000,0.5000,0.000028,0.004827,0.456522,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1469,0.988636,0.296053,0.000000,0.5500,0.000027,0.004858,1.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1470,0.988636,0.210526,0.000000,0.5750,0.000017,0.000882,0.021739,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1471,0.988636,0.217105,0.019608,0.3125,0.000067,0.002709,0.510870,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


# DBSCAN

run a loop to get best epsilon value and minpnts

In [105]:
eps_array = [0.2,0.5,1]
min_samples_array = [5,10,30]

for eps in eps_array:
    for min_samples in min_samples_array:
        clusterer = DBSCAN(eps=eps,min_samples=min_samples).fit(df_scaled)
        cluster_labels = clusterer.labels_

        if len(set(cluster_labels))==1:
            continue
        silhouette_avg = silhouette_score(df_scaled,cluster_labels)       

        print('for eps = ',eps,
             "for min samples = ",min_samples,
             "count of clusters = ",len(set(cluster_labels)),
             "Silhoutte_score = ",silhouette_avg
                    )

for eps =  0.2 for min samples =  5 count of clusters =  29 Silhoutte_score =  0.3398486863888048
for eps =  0.2 for min samples =  10 count of clusters =  20 Silhoutte_score =  0.25270863830104556
for eps =  0.2 for min samples =  30 count of clusters =  5 Silhoutte_score =  0.07969728612512754
for eps =  0.5 for min samples =  5 count of clusters =  42 Silhoutte_score =  0.5296936966529566
for eps =  0.5 for min samples =  10 count of clusters =  27 Silhoutte_score =  0.48148533016330597
for eps =  0.5 for min samples =  30 count of clusters =  11 Silhoutte_score =  0.29512530773935547
for eps =  1 for min samples =  5 count of clusters =  46 Silhoutte_score =  0.5468235232106208
for eps =  1 for min samples =  10 count of clusters =  27 Silhoutte_score =  0.4874332707505682
for eps =  1 for min samples =  30 count of clusters =  11 Silhoutte_score =  0.3000923232422267


# DBSCAN With BestHyperParameters(eps=1,minpnts=5)

In [109]:
dbscan_model = DBSCAN(eps=1,min_samples=5).fit(df_scaled)

print("Clusters",len(set(dbscan_model.labels_)))
print("Score:",silhouette_score(df_scaled,dbscan_model.labels_))     

Clusters 46
Score: 0.5468235232106208


In [110]:
df_movies['dbscan_clusters'] = dbscan_model.labels_

In [111]:
df_movies

,type,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_countries,prod_countries_cnt,main_genre,dbscan_clusters
title,,,,,,,,,,,,
The Three Stooges,SHOW,1934,19,26.0,8.6,1092.0,15.424,7.6,US,1,comedy,0
What's My Line?,SHOW,1950,30,18.0,8.6,1563.0,87.392,6.9,US,1,reality,1
I Love Lucy,SHOW,1951,30,9.0,8.5,25944.0,17.088,8.1,US,1,comedy,0
Mister Rogers' Neighborhood,SHOW,1968,29,31.0,8.7,8675.0,8.747,4.7,US,1,fantasy,34
Lupin the Third,SHOW,1971,23,6.0,7.9,2116.0,45.829,8.0,JP,1,scifi,2
...,...,...,...,...,...,...,...,...,...,...,...,...
Level Playing Field,SHOW,2021,26,1.0,5.5,60.0,4.595,5.0,US,1,documentation,4
Os Ausentes,SHOW,2021,46,1.0,5.9,59.0,4.624,10.0,BR,1,action,-1
Through Our Eyes,SHOW,2021,33,1.0,6.1,38.0,0.840,1.0,US,1,documentation,4


# Movie Recommendation Function

In [112]:
import random

def recommend_movie(movie_name: str):
    movie_name = movie_name.lower()
    df_movies['name'] = df_movies.index.str.lower()

    movie = df_movies[df_movies['name'].str.contains(movie_name,na=False)]

    if not movie.empty:
        cluster = movie['dbscan_clusters'].values[0]
        cluster_movies = df_movies[df_movies['dbscan_clusters']==cluster]

        if len(cluster_movies) >=5:
            recommended_movies = random.sample(list(cluster_movies.index),5)
        else:
            recommended_movies = list(cluster_movies.index)

        print("We can recommend you this movies")
        for m in recommended_movies:
            print(m)
        else:
            print("Movie not found in DataBase")

In [114]:
input_movie = input("Enter movie name")
print("\n")
recommend_movie(input_movie)

Enter movie name The Three Stooges




We can recommend you this movies
Workaholics
Regular Show
Los Espookys
Rowan & Martin's Laugh-In
Sister, Sister
Movie not found in DataBase


In [116]:
import pickle 

pickle_model_path = "movie_recommending_system.pkl"
with open(pickle_model_path,'wb') as f:
    pickle.dump(df_movies,f)
